# 02 · Scratch CNN and imbalance strategies

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = ""
USE_DRIVE = True

if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub", "scikit-learn", "scipy", "pandas", "matplotlib", "pillow", "tqdm"], check=True)
    if USE_DRIVE:
        from google.colab import drive

        drive.mount("/content/drive")
        os.environ.setdefault("XRAY_ARTIFACTS", "/content/drive/MyDrive/xray_project/artifacts")
        os.environ.setdefault("XRAY_RESULTS", "/content/drive/MyDrive/xray_project/results")


def find_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "src" / "config.py").exists():
            return p


ROOT = find_root()
if ROOT is None:
    assert REPO_URL, "Set REPO_URL or open the notebook from inside the repository"
    subprocess.run(["git", "clone", REPO_URL, "xray-vision-pneumonia-CNN"], check=True)
    ROOT = Path("xray-vision-pneumonia-CNN").resolve()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import config as C
from src import dataset as D
from src import evaluate as E
from src import train as T
from src.utils import get_run, load_runs, sync_results_to_repo

C.ensure_dirs()
SEEDS = C.SEEDS
STRATEGIES = ["plain", "class_weight", "oversample", "focal", "augment", "cw_aug"]
EPOCHS, PATIENCE = 40, 8

## Oversampling sanity check

In [ ]:
sp = D.get_splits()
ds, steps = D.make_dataset(sp["train"], training=True, oversample=True)
frac = [float(np.mean(y.numpy())) for _, y in ds.take(40)]
print(f"pneumonia share per batch: min {min(frac):.2f} max {max(frac):.2f} mean {np.mean(frac):.2f}")
assert 0.2 < np.mean(frac) < 0.8

## Strategy comparison (validation, multiple seeds)

In [ ]:
for seed in SEEDS:
    for strategy in STRATEGIES:
        T.train_scratch(strategy, seed, epochs=EPOCHS, patience=PATIENCE)

In [ ]:
runs = load_runs()
base = runs[(runs["family"] == "scratch") & runs["tag"].isna()]
cols = ["val_auc", "val_pr_auc", "val_sensitivity", "val_specificity", "val_spec_at_target_sens", "epochs_trained"]
summary = base.groupby("strategy")[cols].agg(["mean", "std"]).round(4)
summary = summary.sort_values(("val_auc", "mean"), ascending=False)
summary.to_csv(C.RESULTS_DIR / "summary_scratch_strategies.csv")
summary

In [ ]:
full = E.metrics_table(base["run_id"].tolist(), "val").merge(base[["run_id", "strategy", "seed"]], on="run_id")
full.to_csv(C.RESULTS_DIR / "val_metrics_scratch.csv", index=False)
full[["strategy", "seed", "accuracy", "precision", "sensitivity", "specificity", "f1", "roc_auc", "pr_auc"]].round(4)

In [ ]:
BEST = summary.index[0]
print(f"Best strategy by mean validation ROC-AUC: {BEST}")
print(summary.loc[BEST])
best_run = base[(base["strategy"] == BEST) & (base["seed"] == C.SEED)].iloc[0]["run_id"]
E.plot_history(json.load(open(C.HIST_DIR / f"{best_run}.json")), C.FIG_DIR / "curves_scratch_best.png", title=best_run)

## Architecture ablations (best strategy, seed 42)

In [ ]:
ABLATIONS = {
    "depth3": dict(filters=(32, 64, 128)),
    "width_half": dict(filters=(16, 32, 64, 128)),
    "width_double": dict(filters=(64, 128, 256, 512)),
    "no_bn": dict(use_bn=False),
    "dropout0.2": dict(dropout=0.2),
    "dropout0.5": dict(dropout=0.5),
    "flatten": dict(pooling="flatten"),
}
for tag, kw in ABLATIONS.items():
    T.train_scratch(BEST, C.SEED, model_kwargs=kw, tag=f"abl-{tag}", epochs=EPOCHS, patience=PATIENCE)

In [ ]:
runs = load_runs()
abl = runs[runs["tag"].fillna("").str.startswith("abl-")]
ref = runs[(runs["family"] == "scratch") & runs["tag"].isna() & (runs["strategy"] == BEST) & (runs["seed"] == C.SEED)]
show = ["tag", "n_params", "epochs_trained", "val_auc", "val_pr_auc", "val_sensitivity", "val_specificity", "val_spec_at_target_sens"]
ablation = pd.concat([ref.assign(tag="reference"), abl])[show].round(4)
ablation.to_csv(C.RESULTS_DIR / "ablation_baseline.csv", index=False)
ablation

## Augmentation notes

Rotation (±7.2°), zoom (±5%), translation (±2%) and small brightness/contrast changes mimic real variation in patient positioning and exposure. Vertical flips are never used: the heart, stomach bubble and diaphragm have a fixed up-down orientation, so flipped images are anatomically impossible and never occur at test time. Horizontal flips are off in the baseline because they move the heart to the right side; the transfer-learning notebook ablates them explicitly.

In [ ]:
sync_results_to_repo()